In [2]:
import ee
import geemap

# ee.Authenticate()                               # first time only: opens a Google sign-in
ee.Initialize(project="cosmic-ascent-256119") 

county_geom = ee.Geometry.Rectangle([-122.4, 47.4, -122.1, 47.7])

landsat = (ee.ImageCollection("LANDSAT/LC09/C02/T1_L2")
           .filterBounds(county_geom)
           .filterDate("2024-06-01", "2024-08-31")
           .filter(ee.Filter.lt("CLOUD_COVER", 20)))

print("Scenes matching filter:", landsat.size().getInfo())   # Python needs getInfo()

composite = landsat.median().clip(county_geom)
lst = (composite.select("ST_B10").multiply(0.00341802).add(149.0).subtract(273.15)
       .rename("LST_C"))

m = geemap.Map(basemap="Esri.WorldGrayCanvas")
m.centerObject(county_geom, 11)
m.addLayer(composite.select(["SR_B4", "SR_B3", "SR_B2"]).multiply(0.0000275).add(-0.2),
           {"min": 0, "max": 0.3}, "True color (median)")
m.addLayer(lst, {"min": 20, "max": 45, "palette": ["313695", "ffffbf", "a50026"]},
           "Summer LST (deg C)")
m

Scenes matching filter: 5


Map(center=[47.549954682458825, -122.24999999999913], controls=(WidgetControl(options=['position', 'transparen…